In [1]:
# Imports + connexion au drive

import pandas as pd

lien = "https://datasets.imdbws.com/title.ratings.tsv.gz"

ratings = pd.read_csv(lien, sep="\t", na_values="\\N")


In [2]:
# Chargement du fichier TMDb The Movie Database

tmdb = pd.read_csv("../data/TMDB_movie_dataset_v11.csv")

FileNotFoundError: [Errno 2] No such file or directory: '../data/TMDB_movie_dataset_v11.csv'

In [113]:
# Fichier IMDb
# title basics : Une ligne par titre : type, titre, année, durée,genres

import pandas as pd

lien = "https://datasets.imdbws.com/title.basics.tsv.gz"

colonnes = ["tconst", "titleType", "primaryTitle", "isAdult", "startYear", "runtimeMinutes", "genres"]

basics = pd.read_csv(lien, sep="\t", na_values="\\N", usecols=colonnes, low_memory=False)

basics["runtimeMinutes"] = pd.to_numeric(basics["runtimeMinutes"], errors="coerce")

basics["startYear"] = pd.to_numeric(basics["startYear"], errors="coerce")

In [114]:
ratings.head()

,tconst,averageRating,numVotes
0,tt0000001,5.7,2241
1,tt0000002,5.4,330
2,tt0000003,6.4,2412
3,tt0000004,5.0,204
4,tt0000005,6.2,3117


In [115]:
print(basics.shape)
basics.head()

(12979994, 7)


,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,0,1894.0,1.0,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,0,1892.0,5.0,"Animation,Short"
2,tt0000003,short,Poor Pierrot,0,1892.0,5.0,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,0,1892.0,12.0,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,0,1893.0,1.0,Short


In [116]:
df = pd.merge(basics, ratings, on="tconst", how="inner")

In [117]:
df.head()

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes
0,tt0000001,short,Carmencita,0,1894.0,1.0,"Documentary,Short",5.7,2241
1,tt0000002,short,Le clown et ses chiens,0,1892.0,5.0,"Animation,Short",5.4,330
2,tt0000003,short,Poor Pierrot,0,1892.0,5.0,"Animation,Comedy,Romance",6.4,2412
3,tt0000004,short,Un bon bock,0,1892.0,12.0,"Animation,Short",5.0,204
4,tt0000005,short,Blacksmith Scene,0,1893.0,1.0,Short,6.2,3117


In [118]:
print(f"Point de départ : {df.shape[0]} lignes.")

films = df[df["titleType"] == "movie"]
print(f"Filtré sur 'titleType' = 'Movie' : {films.shape[0]} lignes.")

films = films[films["numVotes"] >= 5000]
print(f"Filtré sur 'numVotes' >= 5000 votes : {films.shape[0]} lignes.")

films = films[films["averageRating"] >= 7.5]
print(f"Filtré sur 'averageRating' >= 7.5 : {films.shape[0]} lignes.")

films = films[films["isAdult"] == 0]
print(f"Filtré sur 'isAdult' : {films.shape[0]} lignes.")

films.head()

Point de départ : 1734008 lignes.
Filtré sur 'titleType' = 'Movie' : 352019 lignes.
Filtré sur 'numVotes' >= 5000 votes : 19291 lignes.
Filtré sur 'averageRating' >= 7.5 : 3323 lignes.
Filtré sur 'isAdult' : 3323 lignes.


,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes
2485,tt0006864,movie,Intolerance,0,1916.0,210.0,"Drama,History",7.6,18277
3398,tt0010323,movie,The Cabinet of Dr. Caligari,0,1920.0,67.0,"Horror,Mystery,Thriller",8.0,77573
3967,tt0012349,movie,The Kid,0,1921.0,68.0,"Comedy,Drama,Family",8.2,147895
3972,tt0012364,movie,The Phantom Carriage,0,1921.0,107.0,"Drama,Fantasy,Horror",8.0,16201
4010,tt0012494,movie,Destiny,0,1921.0,114.0,"Drama,Fantasy,Horror",7.6,7525


In [120]:
mes_films = set(films["tconst"]) 
morceaux_gardes = [] 
lien = "https://datasets.imdbws.com/title.akas.tsv.gz" 

for morceau in pd.read_csv(lien, sep="\t", na_values="\\N",
                           usecols=["titleId", "title", "region"],
                           chunksize=2_000_000):
    est_fr = morceau["region"] == "FR"    
    est_a_moi = morceau["titleId"].isin(mes_films)    
    garde = morceau[est_fr & est_a_moi]    
    morceaux_gardes.append(garde) 
    
titres_fr = pd.concat(morceaux_gardes).drop_duplicates(subset="titleId")

In [121]:
# Fusion de notre liste de films filtrée avec la base akas pour intégrer les titres français.
films = pd.merge(films, titres_fr, 
                    left_on="tconst",
                    right_on="titleId",
                      how="left")

print(f"{films.shape[0]} lignes et {films.shape[1]} colonnes.")
films.head()

3323 lignes et 12 colonnes.


,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,titleId,title,region
0,tt0006864,movie,Intolerance,0,1916.0,210.0,"Drama,History",7.6,18277,tt0006864,Intolérance,FR
1,tt0010323,movie,The Cabinet of Dr. Caligari,0,1920.0,67.0,"Horror,Mystery,Thriller",8.0,77573,tt0010323,Le Cabinet du docteur Caligari,FR
2,tt0012349,movie,The Kid,0,1921.0,68.0,"Comedy,Drama,Family",8.2,147895,tt0012349,Le Kid,FR
3,tt0012364,movie,The Phantom Carriage,0,1921.0,107.0,"Drama,Fantasy,Horror",8.0,16201,tt0012364,La charrette fantôme,FR
4,tt0012494,movie,Destiny,0,1921.0,114.0,"Drama,Fantasy,Horror",7.6,7525,tt0012494,Les Trois Lumières,FR


In [122]:
# Ajout des titres "primaryTitle" dans les titres vides de la colonne "title"
films["title"] = films["title"].fillna(films["primaryTitle"])

films["title"].isna().sum()             # ok plus de vide dans "title"

np.int64(0)

In [123]:
# Suppression des colonnes inutiles = "region" ne contient que FR
print(films["region"].value_counts())       # FR = 2705 lignes
print(films.shape)
films = films.drop(columns="region")
print(films.shape)                          # ok colonne supprimée

region
FR    2705
Name: count, dtype: int64
(3323, 12)
(3323, 11)


In [124]:
# Suppression des colonnes inutiles = "titleId" doublon avec colonne "tconst"
films = films.drop(columns="titleId")
print(films.shape)                          # ok colonne supprimée

(3323, 10)


In [125]:
# Vérification de doublons 
films.duplicated().sum()

np.int64(0)

In [126]:
films["titleType"].value_counts()

titleType
movie    3323
Name: count, dtype: int64

In [127]:
# Renomage des colonnes
films = films.rename(columns={"primaryTitle": "titre_international", "startYear": "annee_sortie", "runtimeMinutes": "duree_min", "averageRating": "note_moyenne", "numVotes": "nombre_votes", "title": "titre_affiche"})

In [ ]:
# Suppression colonnes titleType et isAdult
films = films.drop(columns=["titleType", "isAdult"])

In [129]:
# Formatage des colonnes duree_min et annee_sortie en Int64
films["duree_min"] = films["duree_min"].astype("Int64")
films["annee_sortie"] = films["annee_sortie"].astype("Int64")

print(films["duree_min"].isna().sum())
print(films["annee_sortie"].isna().sum())

0
0


In [130]:
films = films[["tconst", "titre_international", "titre_affiche", "annee_sortie", "duree_min", "genres", "note_moyenne", "nombre_votes"]]
films.head()

,tconst,titre_international,titre_affiche,annee_sortie,duree_min,genres,note_moyenne,nombre_votes
0,tt0006864,Intolerance,Intolérance,1916,210,"Drama,History",7.6,18277
1,tt0010323,The Cabinet of Dr. Caligari,Le Cabinet du docteur Caligari,1920,67,"Horror,Mystery,Thriller",8.0,77573
2,tt0012349,The Kid,Le Kid,1921,68,"Comedy,Drama,Family",8.2,147895
3,tt0012364,The Phantom Carriage,La charrette fantôme,1921,107,"Drama,Fantasy,Horror",8.0,16201
4,tt0012494,Destiny,Les Trois Lumières,1921,114,"Drama,Fantasy,Horror",7.6,7525


In [131]:
films.describe()

,annee_sortie,duree_min,note_moyenne,nombre_votes
count,3323.0,3323.0,3323.000000,3.323000e+03
mean,1994.060187,121.04183,7.835239,1.406747e+05
std,25.961568,31.54104,0.314368,3.040038e+05
min,1916.0,45.0,7.500000,5.000000e+03
25%,1976.0,100.0,7.600000,9.818500e+03
50%,2003.0,116.0,7.800000,2.378400e+04
75%,2015.0,137.0,8.000000,1.043450e+05
max,2026.0,566.0,9.800000,3.266351e+06


In [132]:
films["tconst"].duplicated().sum()

np.int64(0)

In [134]:
films[(films["tconst"] == "tt0111161")]

,tconst,titre_international,titre_affiche,annee_sortie,duree_min,genres,note_moyenne,nombre_votes
1226,tt0111161,The Shawshank Redemption,Les Évadés,1994,142,Drama,9.3,3266351


In [135]:
# Analyse du catalgoue TMDb
print(f"{tmdb.shape[0]} lignes et {tmdb.shape[1]} colonnes.")

1529949 lignes et 24 colonnes.


In [136]:
# Suppression des lignes sans clé IMDb (qui ne pourront donc pas intégrer ntore catalogue)
print(tmdb["imdb_id"].isna().sum())
tmdb = tmdb.dropna(subset="imdb_id")
print(tmdb["imdb_id"].isna().sum())

836561
0


In [137]:
print(tmdb["imdb_id"].duplicated().sum())

tmdb = tmdb[tmdb["imdb_id"].isin(films["tconst"])]
print(tmdb.shape)

5633
(3230, 24)


In [ ]:
# Voir les doublons
doublons = tmdb[tmdb["imdb_id"].duplicated(keep=False)].sort_values("imdb_id")
doublons[["imdb_id", "title", "vote_count", "status", "release_date", "poster_path"]]
doublons.shape

(7, 24)

In [ ]:
# Supprimer les doublons : garder la ligne avec le plus de votes
tmdb = tmdb.sort_values("vote_count", ascending=False)
tmdb = tmdb.drop_duplicates(subset="imdb_id", keep="first")
print(tmdb["imdb_id"].duplicated().sum())

In [141]:
# Colonnes utiles
tmdb = tmdb[["imdb_id", "original_language", "production_countries",
             "production_companies", "overview", "poster_path"]]
tmdb.head()

,imdb_id,original_language,production_countries,production_companies,overview,poster_path
0,tt1375666,en,"United Kingdom, United States of America","Legendary Pictures, Syncopy, Warner Bros. Pict...","Cobb, a skilled thief who commits corporate es...",/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg
1,tt0816692,en,"United Kingdom, United States of America","Legendary Pictures, Syncopy, Lynda Obst Produc...",The adventures of a group of explorers who mak...,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg
2,tt0468569,en,"United Kingdom, United States of America","DC Comics, Legendary Pictures, Syncopy, Isobel...",Batman raises the stakes in his war on crime. ...,/qJ2tW6WMUDux911r6m7haRef0WH.jpg
3,tt0499549,en,"United States of America, United Kingdom","Dune Entertainment, Lightstorm Entertainment, ...","In the 22nd century, a paraplegic Marine is di...",/kyeqWdyUXW608qlYkRqosgbbJyK.jpg
4,tt0848228,en,United States of America,Marvel Studios,When an unexpected enemy emerges and threatens...,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg


In [ ]:
# Jointure
films = pd.merge(films, tmdb, left_on="tconst", right_on="imdb_id", how="left")
print(len(films))

3323


In [143]:
# Supprimer la clé en double
films = films.drop(columns="imdb_id")

In [144]:
# Films sans correspondance TMDB
print(films["poster_path"].isna().sum())

143


In [145]:
# URL complète de l'affiche
films["affiche_url"] = "https://image.tmdb.org/t/p/w500" + films["poster_path"]

In [153]:
# Films français (règle : la France figure parmi les pays de production)
films["film_francais"] = films["production_countries"].str.contains("France", na=False)
print(films["film_francais"].sum())
print(f"{films["film_francais"].mean():.1%} du catalogue propose des films de production française (coproductions incluses).")
print(len(films))

397
11.9% du catalogue propose des films de production française (coproductions incluses).
3323


In [157]:
# title.principals par morceaux : nos films + acteurs, actrices, réalisateurs
mes_films = set(films["tconst"])
morceaux = []
lien = "https://datasets.imdbws.com/title.principals.tsv.gz"

for morceau in pd.read_csv(lien, sep="\t", na_values="\\N",
                           usecols=["tconst", "ordering", "nconst", "category"],
                           chunksize=2_000_000):
    est_a_moi = morceau["tconst"].isin(mes_films)
    bon_role = morceau["category"].isin(["actor", "actress", "director"])
    morceaux.append(morceau[est_a_moi & bon_role])

principals = pd.concat(morceaux)
print(principals.shape)

(34481, 4)


In [ ]:
# name.basics par morceaux : seulement les personnes de nos films
mes_personnes = set(principals["nconst"])
morceaux = []
lien = "https://datasets.imdbws.com/name.basics.tsv.gz"

for morceau in pd.read_csv(lien, sep="\t", na_values="\\N",
                           usecols=["nconst", "primaryName"],
                           chunksize=2_000_000):
    morceaux.append(morceau[morceau["nconst"].isin(mes_personnes)])

noms = pd.concat(morceaux)

# Ajouter les noms aux rôles
principals = pd.merge(principals, noms, on="nconst", how="left")
principals = principals.dropna(subset="primaryName")
principals = principals.sort_values(["tconst", "ordering"])

In [160]:
# Une ligne par film : les X premiers acteurs, et les réalisateurs
acteurs = principals[principals["category"].isin(["actor", "actress"])]
acteurs = acteurs.groupby("tconst").head(3)          # combien d'acteurs principaux ?
acteurs = acteurs.groupby("tconst")["primaryName"].agg(", ".join).reset_index()
acteurs = acteurs.rename(columns={"primaryName": "acteurs"})

realisateurs = principals[principals["category"] == "director"]
realisateurs = realisateurs.groupby("tconst")["primaryName"].agg(", ".join).reset_index()
realisateurs = realisateurs.rename(columns={"primaryName": "realisateurs"})

# Jointure avec le catalogue
films = pd.merge(films, acteurs, on="tconst", how="left")
films = pd.merge(films, realisateurs, on="tconst", how="left")
print(len(films))
print(films[["acteurs", "realisateurs"]].isna().sum())
films[["titre_affiche", "acteurs", "realisateurs"]].head()

3323
acteurs         250
realisateurs      5
dtype: int64


,titre_affiche,acteurs,realisateurs
0,Intolérance,"Lillian Gish, Lillian Gish, Robert Harron",D.W. Griffith
1,Le Cabinet du docteur Caligari,"Werner Krauss, Conrad Veidt, Friedrich Feher",Robert Wiene
2,Le Kid,"Charles Chaplin, Edna Purviance, Jackie Coogan",Charles Chaplin
3,La charrette fantôme,"Victor Sjöström, Hilda Borgström, Tore Svennberg",Victor Sjöström
4,Les Trois Lumières,"Bernhard Goetzke, Bernhard Goetzke, Bernhard G...",Fritz Lang


In [3]:
# Sauvegarde
films.to_csv("../catalogue/films.csv", index=False) # pour partage avec le groupe projet

NameError: name 'films' is not defined